# Saberly Research: Optimizing Gemma 4  for Pedagogical Tutoring
Este notebook documenta el proceso de experimentación para configurar a **Gemma 4** como el tutor oficial de Saberly. El objetivo es garantizar respuestas precisas para las pruebas ICFES en Colombia, evitando alucinaciones y manteniendo un tono motivador.


In [ ]:
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

# Sección 1: Configuración del Chat Template y Tokenización
Para que Gemma 4  se comporte como un tutor pedagógico, es vital configurar el `chat_template`. 
Esto asegura que el modelo distinga correctamente entre las instrucciones del sistema (System Prompt), 
las dudas del estudiante y sus propias respuestas. Sin esta configuración, el modelo podría confundir los roles 
y perder el hilo de la tutoría.

In [ ]:
model_id = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b/1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.chat_template = "{{ bos_token }}{% if messages[0]['role'] == 'system' %}{{ '<start_of_turn>system\n' + messages[0]['content'] + '<end_of_turn>\n' }}{% set loop_messages = messages[1:] %}{% else %}{% set loop_messages = messages %}{% endif %}{% for message in loop_messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant...') }}{% endif %}{{ '<start_of_turn>' + message['role'] + '\n' + message['content'] | trim + '<end_of_turn>\n' }}{% endfor %}{% if add_generation_prompt %}{{ '<start_of_turn>model\n' }}{% endif %}"

# Sección 2: Optimización de Parámetros de Inferencia
En esta sección, ajustamos los hiperparámetros de generación para equilibrar la creatividad y la precisión:
* **Temperature (0.2):** Reducimos la aleatoriedad para evitar que el modelo invente datos falsos (alucinaciones) en temas de ciencia.
* **Repetition Penalty (1.3):** Obligamos al modelo a usar un vocabulario variado, evitando que se quede en bucles de frases cordiales.
* **Top_p (0.85):** Limitamos el muestreo a las palabras más probables para mantener la coherencia lógica.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

mensajes_chat = [
    {
        "role": "user", 
        "content": """Eres el Tutor IA de Saberly. Ayudas a estudiantes colombianos con el ICFES.
        Si un estudiante se equivoca, no le des la respuesta directa. 
        Explica el concepto físico o matemático y relaciónalo con algo cotidiano. Usa un tono motivador."""
    },
    {
        "role": "model", 
        "content": "¡Hola! Soy tu tutor de Saberly. Entiendo tu duda sobre los valores de V. Analicemos juntos: ¿Qué relación crees que existe entre V y el resultado final según la fórmula del problema?"
    },
    {
        "role": "user", 
        "content": "Es que pensé que a mayor V, mayor sería el resultado, por eso marqué la C."
    }
]


## Sección 3: Optimización de Parámetros de Inferencia
Ajustamos los hiperparámetros para evitar alucinaciones:
* **Temperature (0.2):** Para mayor precisión.
* **Repetition Penalty (1.3):** Para evitar bucles de texto.

In [ ]:
prompt = tokenizer.apply_chat_template(mensajes_chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

stop_token_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,      # Bajamos un poco para evitar que divague demasiado
        do_sample=True,
        temperature=0.2,         # MUCHO más bajo para evitar que invente JSONs
        repetition_penalty=1.3,  # Subimos para que no repita patrones de entrenamiento
        top_p=0.85,              # Filtramos palabras poco probables
        eos_token_id=stop_token_id,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True
    )


## Sección 4: Algoritmo de Limpieza y Post-procesamiento
Implementamos filtros de seguridad para limpiar tokens especiales y asegurar que la respuesta entregada al estudiante sea puramente pedagógica y libre de ruido técnico.

In [ ]:
full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
respuesta_modelo = full_text.split("<start_of_turn>model\n")[-1]

# LIMPIEZA EXTRA: Si el modelo empieza a hablar en formato JSON, cortamos ahí.
if '{"' in respuesta_modelo:
    respuesta_modelo = respuesta_modelo.split('{"')[0]

# Limpiar etiquetas y tokens
respuesta_modelo = re.sub(r'<.*?>', '', respuesta_modelo)
for tag in ["<end_of_turn>", "<start_of_turn>", "user\n", "model\n"]:
    respuesta_modelo = respuesta_modelo.replace(tag, "")

print(f"Tutor Saberly: {respuesta_modelo.strip()}")] para generar una respuesta y devolverla como un JSON string."
] para generar una respuesta y devolverla como un JSON string."

## Conclusión: 
Estos experimentos validan que Gemma 4  es capaz de actuar como un tutor reflexivo. Estos parámetros y lógicas de limpieza fueron los que finalmente se integraron en el backend de producción de Saberly para garantizar una experiencia educativa de alta calidad en el municipio de Pinillos.